In [2]:
# ============================================================
# CYCLISTIC BIKE-SHARE CASE STUDY
# Google Data Analytics Capstone — Case Study 1
# Tool: Python | Data: Divvy 2019 Q1 & 2020 Q1
# ============================================================
# PHASES: Ask → Prepare → Process → Analyze → Share → Act
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os

# ── Aesthetic config ─────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted")
COLORS = {"member": "#1A73E8", "casual": "#E8710A"}
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# PHASE 1 — ASK
# Business task: Understand how annual members and casual
# riders use Cyclistic bikes differently, in order to design
# a marketing strategy that converts casual riders into
# annual members.
# ============================================================
print("=" * 60)
print("CYCLISTIC BIKE-SHARE CASE STUDY")
print("Business Task: How do members and casual riders differ?")
print("=" * 60)


# ============================================================
# PHASE 2 — PREPARE
# Load the two quarterly datasets and inspect them.
# ============================================================
print("\n📂 [PREPARE] Loading data...")

df_2019 = pd.read_csv("Divvy_Trips_2019_Q1.csv")
df_2020 = pd.read_csv("Divvy_Trips_2020_Q1.csv")

print(f"  2019 Q1 shape : {df_2019.shape}")
print(f"  2020 Q1 shape : {df_2020.shape}")
print("\n  2019 columns :", list(df_2019.columns))
print("  2020 columns :", list(df_2020.columns))


# ============================================================
# PHASE 3 — PROCESS
# Rename columns so both datasets share the same schema,
# combine them, engineer features, and clean the data.
# ============================================================
print("\n🔧 [PROCESS] Cleaning & transforming data...")

# 3a. Rename 2019 columns to match 2020 schema
df_2019 = df_2019.rename(columns={
    "trip_id":           "ride_id",
    "bikeid":            "rideable_type",
    "start_time":        "started_at",
    "end_time":          "ended_at",
    "from_station_name": "start_station_name",
    "from_station_id":   "start_station_id",
    "to_station_name":   "end_station_name",
    "to_station_id":     "end_station_id",
    "usertype":          "member_casual",
})

# 3b. Harmonise the member_casual labels
df_2019["member_casual"] = df_2019["member_casual"].replace(
    {"Subscriber": "member", "Customer": "casual"}
)

# 3c. Keep only the columns that exist in both datasets
shared_cols = [
    "ride_id", "started_at", "ended_at",
    "start_station_name", "start_station_id",
    "end_station_name",   "end_station_id",
    "member_casual",
]
df_2019 = df_2019[shared_cols]
df_2020 = df_2020[shared_cols]

# 3d. Combine into one dataframe
df = pd.concat([df_2019, df_2020], ignore_index=True)
print(f"  Combined rows : {len(df):,}")

# 3e. Parse datetimes
df["started_at"] = pd.to_datetime(df["started_at"])
df["ended_at"]   = pd.to_datetime(df["ended_at"])

# 3f. Engineer ride_length (minutes) and day_of_week
df["ride_length"] = (df["ended_at"] - df["started_at"]).dt.total_seconds() / 60
df["day_of_week"] = df["started_at"].dt.day_name()
df["month"]       = df["started_at"].dt.month_name()
df["hour"]        = df["started_at"].dt.hour

# 3g. Remove bad rows (negative or zero ride lengths)
before = len(df)
df = df[df["ride_length"] > 0].copy()
print(f"  Removed {before - len(df):,} rows with non-positive ride_length")
print(f"  Clean dataset : {len(df):,} rows")

# 3h. Document null counts
print("\n  Null counts per column:")
print(df.isnull().sum().to_string())


# ============================================================
# PHASE 4 — ANALYZE
# Descriptive statistics and trend identification.
# ============================================================
print("\n📊 [ANALYZE] Running descriptive analysis...")

# 4a. Overall ride_length stats by user type
stats = (
    df.groupby("member_casual")["ride_length"]
    .agg(count="count", mean="mean", median="median", max="max", std="std")
    .round(2)
)
print("\n  Ride length summary (minutes):")
print(stats.to_string())

# 4b. Average ride length by day of week
day_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
avg_by_day = (
    df.groupby(["member_casual","day_of_week"])["ride_length"]
    .mean()
    .reset_index()
)
avg_by_day["day_of_week"] = pd.Categorical(avg_by_day["day_of_week"], categories=day_order, ordered=True)
avg_by_day = avg_by_day.sort_values("day_of_week")

# 4c. Number of rides by day of week
count_by_day = (
    df.groupby(["member_casual","day_of_week"])
    .size()
    .reset_index(name="ride_count")
)
count_by_day["day_of_week"] = pd.Categorical(count_by_day["day_of_week"], categories=day_order, ordered=True)
count_by_day = count_by_day.sort_values("day_of_week")

# 4d. Rides by hour of day
count_by_hour = (
    df.groupby(["member_casual","hour"])
    .size()
    .reset_index(name="ride_count")
)

print("\n  Average ride length by day of week:")
print(avg_by_day.pivot(index="day_of_week", columns="member_casual", values="ride_length").to_string())


# ============================================================
# PHASE 5 — SHARE
# Create polished visualisations and save them.
# ============================================================
print("\n🎨 [SHARE] Creating visualizations...")

def save(fig, name):
    path = os.path.join(OUTPUT_DIR, name)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print(f"  ✅ Saved → {path}")
    plt.close(fig)


# ── Chart 1: Total rides by user type ────────────────────────
ride_counts = df["member_casual"].value_counts()
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(ride_counts.index, ride_counts.values,
              color=[COLORS[k] for k in ride_counts.index], width=0.5)
ax.bar_label(bars, fmt="{:,.0f}", padding=5)
ax.set_title("Total Rides by User Type", fontsize=14, fontweight="bold")
ax.set_xlabel("User Type"); ax.set_ylabel("Number of Rides")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
save(fig, "01_total_rides_by_user_type.png")


# ── Chart 2: Average ride length by user type ────────────────
avg_length = df.groupby("member_casual")["ride_length"].mean()
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(avg_length.index, avg_length.values,
              color=[COLORS[k] for k in avg_length.index], width=0.5)
ax.bar_label(bars, fmt="{:.1f} min", padding=5)
ax.set_title("Average Ride Length by User Type", fontsize=14, fontweight="bold")
ax.set_xlabel("User Type"); ax.set_ylabel("Avg Ride Length (minutes)")
save(fig, "02_avg_ride_length_by_user_type.png")


# ── Chart 3: Rides by day of week ────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
for utype, grp in count_by_day.groupby("member_casual"):
    ax.plot(grp["day_of_week"], grp["ride_count"], marker="o",
            label=utype.capitalize(), color=COLORS[utype], linewidth=2)
ax.set_title("Number of Rides by Day of Week", fontsize=14, fontweight="bold")
ax.set_xlabel("Day of Week"); ax.set_ylabel("Number of Rides")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.legend(); fig.autofmt_xdate()
save(fig, "03_rides_by_day_of_week.png")


# ── Chart 4: Avg ride length by day of week ──────────────────
fig, ax = plt.subplots(figsize=(10, 5))
for utype, grp in avg_by_day.groupby("member_casual"):
    ax.plot(grp["day_of_week"], grp["ride_length"], marker="o",
            label=utype.capitalize(), color=COLORS[utype], linewidth=2)
ax.set_title("Average Ride Length by Day of Week", fontsize=14, fontweight="bold")
ax.set_xlabel("Day of Week"); ax.set_ylabel("Avg Ride Length (minutes)")
ax.legend(); fig.autofmt_xdate()
save(fig, "04_avg_ride_length_by_day.png")


# ── Chart 5: Rides by hour of day ────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
for utype, grp in count_by_hour.groupby("member_casual"):
    ax.plot(grp["hour"], grp["ride_count"], marker="o",
            label=utype.capitalize(), color=COLORS[utype], linewidth=2)
ax.set_title("Rides by Hour of Day", fontsize=14, fontweight="bold")
ax.set_xlabel("Hour (0 = midnight)"); ax.set_ylabel("Number of Rides")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.set_xticks(range(0, 24))
ax.legend()
save(fig, "05_rides_by_hour_of_day.png")


# ── Chart 6: Ride length distribution (box plot) ─────────────
# Cap at 120 min for readability
df_cap = df[df["ride_length"] <= 120]
fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(data=df_cap, x="member_casual", y="ride_length",
            palette=COLORS, ax=ax, width=0.4)
ax.set_title("Ride Length Distribution (capped at 120 min)", fontsize=14, fontweight="bold")
ax.set_xlabel("User Type"); ax.set_ylabel("Ride Length (minutes)")
save(fig, "06_ride_length_distribution.png")


# ── Export summary CSV ────────────────────────────────────────
summary = df.groupby("member_casual").agg(
    total_rides   = ("ride_id",     "count"),
    avg_duration  = ("ride_length", "mean"),
    median_duration=("ride_length", "median"),
    max_duration  = ("ride_length", "max"),
).round(2)
summary_path = os.path.join(OUTPUT_DIR, "summary_stats.csv")
summary.to_csv(summary_path)
print(f"  ✅ Summary CSV saved → {summary_path}")


# ============================================================
# PHASE 6 — ACT
# Key findings and top 3 recommendations.
# ============================================================
print("\n" + "=" * 60)
print("📋 [ACT] KEY FINDINGS & RECOMMENDATIONS")
print("=" * 60)

print("""
KEY FINDINGS
────────────
1. Members take more rides overall, but casual riders take
   significantly LONGER rides on average.

2. Casual riders peak on WEEKENDS → leisure use.
   Members peak on WEEKDAYS (morning & evening rush hours)
   → commuting behaviour.

3. Members show strong 8 AM and 5 PM ride spikes, consistent
   with a daily commute pattern.

4. Casual riders' average trip duration is roughly 2–3×
   longer than members', suggesting recreational use.

TOP 3 RECOMMENDATIONS
─────────────────────
1. WEEKEND MEMBERSHIP PROMOTION
   Target casual riders on Saturdays and Sundays with
   targeted digital ads highlighting the cost savings of
   an annual membership vs. paying per ride.

2. COMMUTER CONVERSION CAMPAIGN
   Show casual riders how much they could save by using
   Cyclistic to commute. Use ride history data to identify
   casual riders who already ride on weekdays and send them
   personalised offers.

3. TRIAL / SEASONAL MEMBERSHIP TIER
   Introduce a discounted "Summer Pass" or trial membership
   that lowers the barrier to commitment. Once riders
   experience membership benefits, full conversion is more
   likely.
""")

print("✅ Case study complete!")

CYCLISTIC BIKE-SHARE CASE STUDY
Business Task: How do members and casual riders differ?

📂 [PREPARE] Loading data...
  2019 Q1 shape : (365069, 12)
  2020 Q1 shape : (426887, 13)

  2019 columns : ['trip_id', 'start_time', 'end_time', 'bikeid', 'tripduration', 'from_station_id', 'from_station_name', 'to_station_id', 'to_station_name', 'usertype', 'gender', 'birthyear']
  2020 columns : ['ride_id', 'rideable_type', 'started_at', 'ended_at', 'start_station_name', 'start_station_id', 'end_station_name', 'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng', 'member_casual']

🔧 [PROCESS] Cleaning & transforming data...
  Combined rows : 791,956
  Removed 210 rows with non-positive ride_length
  Clean dataset : 791,746 rows

  Null counts per column:
ride_id               0
started_at            0
ended_at              0
start_station_name    0
start_station_id      0
end_station_name      0
end_station_id        0
member_casual         0
ride_length           0
day_of_week      

/var/folders/ft/c9zpjvhd6s1f5clcy55n7rmc0000gn/T/ipykernel_73246/2232938349.py:231: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df_cap, x="member_casual", y="ride_length",


  ✅ Saved → outputs/06_ride_length_distribution.png
  ✅ Summary CSV saved → outputs/summary_stats.csv

📋 [ACT] KEY FINDINGS & RECOMMENDATIONS

KEY FINDINGS
────────────
1. Members take more rides overall, but casual riders take
   significantly LONGER rides on average.

2. Casual riders peak on WEEKENDS → leisure use.
   Members peak on WEEKDAYS (morning & evening rush hours)
   → commuting behaviour.

3. Members show strong 8 AM and 5 PM ride spikes, consistent
   with a daily commute pattern.

4. Casual riders' average trip duration is roughly 2–3×
   longer than members', suggesting recreational use.

TOP 3 RECOMMENDATIONS
─────────────────────
1. WEEKEND MEMBERSHIP PROMOTION
   Target casual riders on Saturdays and Sundays with
   targeted digital ads highlighting the cost savings of
   an annual membership vs. paying per ride.

2. COMMUTER CONVERSION CAMPAIGN
   Show casual riders how much they could save by using
   Cyclistic to commute. Use ride history data to identify
   casua